<a href="https://colab.research.google.com/github/Naufallm/RAG-Quantitative-Research-Engine-for-IDX/blob/main/DSC_Project_IDX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Member
- Ahmad Naufal Luthfan Marzuqi
- Syahrial Nur Faturrahman

# Step 1: Install Library
Pada tahap ini, kita menginstal library `pypdf` untuk membaca file PDF dan `langchain` untuk memproses teks.
Library `pandas` digunakan untuk manajemen data tabel jika diperlukan.

Pada Step ini akan diminta untuk restart runtime, setelah restart dapat dijalankan kembali mulai dari awal ya!


In [ ]:
!pip install -U pypdf langchain langchain-community langchain-text-splitters pandas

# Step 2: Sinkronisasi Data
Kita mengambil dataset laporan keuangan yang sudah dikumpulkan di GitHub agar bisa diproses langsung di environment Colab ini.


Untuk link github nya https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX

In [1]:
!git clone https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX.git

Cloning into 'RAG-Quantitative-Research-Engine-for-IDX'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 91 (delta 20), reused 54 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 27.60 MiB | 20.36 MiB/s, done.
Resolving deltas: 100% (20/20), done.


# Step 3: Verifikasi File PDF
Tahap ini memastikan bahwa folder `dataset_idx` telah terunduh dengan benar dan menghitung jumlah file yang siap diproses.

In [ ]:
import os

dataset_path = "/content/RAG-Quantitative-Research-Engine-for-IDX/dataset_idx"
pdf_files = [f for f in os.listdir(dataset_path) if f.endswith(".pdf")]
print("Jumlah file PDF:", len(pdf_files))
print("\nDaftar file PDF: ")
for file in pdf_files:
  print(file)

Jumlah file PDF: 50

Daftar file PDF: 
FinancialStatement-2026-I-RISE.pdf
FinancialStatement-2026-I-ARTO.pdf
FinancialStatement-2026-I-BBCA.pdf
FinancialStatement-2026-I-ESIP.pdf
FinancialStatement-2026-I-SONA.pdf
FinancialStatement-2026-I-SBMA.pdf
FinancialStatement-2026-I-IBFN.pdf
FinancialStatement-2026-I-GIAA.pdf
FinancialStatement-2026-I-EAST.pdf
FinancialStatement-2026-I-AUTO.pdf
FinancialStatement-2026-I-FORE.pdf
FinancialStatement-2026-I-BLOG.pdf
FinancialStatement-2026-I-BMRI.pdf
FinancialStatement-2026-I-PGAS.pdf
FinancialStatement-2026-I-DSNG.pdf
FinancialStatement-2026-I-ISSP.pdf
FinancialStatement-2026-I-MORA.pdf
FinancialStatement-2026-I-BNLI.pdf
FinancialStatement-2026-I-PMJS.pdf
FinancialStatement-2026-I-EDGE.pdf
FinancialStatement-2026-I-DCII.pdf
FinancialStatement-2026-I-MYTX.pdf
FinancialStatement-2026-I-BOLT.pdf
FinancialStatement-2026-I-PGLI.pdf
FinancialStatement-2026-I-IKPM.pdf
FinancialStatement-2026-I-CFIN.pdf
FinancialStatement-2026-I-MCOL.pdf
FinancialStateme

# Step 4: PDF Text Extraction
Di sini kita membaca setiap halaman PDF dan mengubahnya menjadi teks string.
Kita juga mengambil Nama Perusahaan (Ticker) dari nama file menggunakan fungsi split teks.

In [ ]:
from pypdf import PdfReader
from tqdm import tqdm
import os

all_raw_documents = []

total_files = len(pdf_files)

for filename in tqdm(pdf_files, desc="Memproses PDF", unit="file"):
    path = os.path.join(dataset_path, filename)

    try:
        reader = PdfReader(path)
        full_text = ""

        for page in reader.pages:
            page_content = page.extract_text()
            if page_content:
                full_text += page_content + "\n"

        ticker_name = filename.split("-")[-1].replace(".pdf", "")

        all_raw_documents.append({
            "ticker": ticker_name,
            "filename": filename,
            "text": full_text
        })

    except Exception as e:
        print(f"Gagal membaca {filename}: {e}")

print(f"Proses selesai. Berhasil mengekstrak {len(all_raw_documents)} dokumen.")

Memproses PDF: 100%|██████████| 50/50 [04:03<00:00,  4.88s/file]

Proses selesai. Berhasil mengekstrak 50 dokumen.


# Step 5: Hierarchical Chunking
Kita menggunakan paket 'langchain_text_splitters' yang merupakan standar terbaru dari LangChain.
Teks laporan keuangan akan dipecah menjadi bagian-bagian kecil agar dapat diproses secara efisien
oleh model bahasa (Groq/LLM).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

final_chunks = []

for doc in tqdm(all_raw_documents, desc="Membuat chunks", unit="dokumen"):
    chunks = text_splitter.split_text(doc["text"])

    for chunk in chunks:
        final_chunks.append({
            "ticker": doc["ticker"],
            "source": doc["filename"],
            "content": chunk
        })

print(f"Total chunks yang dihasilkan: {len(final_chunks)}")

# Simpan hasil chunks ke file lokal
with open("final_chunks.json", "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=2)

print("Chunks berhasil disimpan ke final_chunks.json")

Membuat chunks: 100%|██████████| 50/50 [00:00<00:00, 132.71dokumen/s]


Total chunks yang dihasilkan: 9744
Chunks berhasil disimpan ke final_chunks.json


# Step 6: Quality Control & Sampel Output
Tahap terakhir adalah memastikan data hasil potongan (chunk) memiliki metadata yang benar
dan menampilkan sampel isi datanya untuk bahan laporan checkpoint.

In [ ]:
print(f"Total File Berhasil Diproses : {len(all_raw_documents)}")
print(f"Total Potongan Teks (Chunks) : {len(final_chunks)}")
print("-" * 40)

if len(final_chunks) > 0:
    print(f"SAMPEL DATA DARI EMITEN     : {final_chunks[0]['ticker']}")
    print(f"SUMBER FILE ASLI            : {final_chunks[0]['source']}")
    print("-" * 40)
    print("CUPLIKAN ISI TEKS:")
    print(final_chunks[0]['content'][:600] + "...")

Total File Berhasil Diproses : 50
Total Potongan Teks (Chunks) : 9744
----------------------------------------
SAMPEL DATA DARI EMITEN     : RISE
SUMBER FILE ASLI            : FinancialStatement-2026-I-RISE.pdf
----------------------------------------
CUPLIKAN ISI TEKS:
Perseroan dengan ini menyampaikan laporan keuangan untuk periode 3 Bulan yang berakhir pada 31/03/2026 dengan ikhtisar sebagai berikut : 
  
Informasi mengenai anak perusahaan Perseroan sebagai berikut : 
  
  
Nomor Surat 104/SKE/CS/JSMS/IV/2026
Nama Emiten PT Jaya Sukses Makmur Sentosa Tbk.
Kode Emiten RISE
Perihal Penyampaian Laporan Keuangan Interim Yang Tidak Diaudit
No Nama Kegiatan
Usaha
Lokasi Tahun
Komersil
Status
Operasi
Jumlah Aset Satuan Mata
Uang
Persentase
(%)
1 PT Platinum
Surya Abadi
Sentosa
Real Estate Surabaya Non Operating 41.541 JUTAAN IDR 50.0
2 PT Tanrise
Makmur
Sentosa
R...


# Step 7: Instalasi Library Vector Database & Embedding
Kita menginstal `chromadb` untuk penyimpanan data vektor dan `langchain-huggingface` untuk mengubah teks menjadi angka koordinat (embedding).

In [6]:
!pip install -q chromadb sentence-transformers langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38

# Step 8: Load Data Chunks

1.   List item
2.   List item


Alih-alih mengulang proses PDF yang berat, kita memuat kembali 9.744 potongan teks (chunks) yang sudah disimpan ke dalam file `final_chunks.json` pada Checkpoint 2.

In [13]:
import json
import os

# Path menuju file JSON di repository github yang sudah di-clone
json_path = "RAG-Quantitative-Research-Engine-for-IDX/final_chunks.json"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        loaded_chunks = json.load(f)
    print(f"✅ Berhasil memuat {len(loaded_chunks)} chunks dari file JSON.")
else:
    print("❌ Error: File JSON tidak ditemukan. Periksa path repository Anda.")

✅ Berhasil memuat 9744 chunks dari file JSON.


# Step 9: Setup Multilingual Embedding Model
Kita menggunakan model `paraphrase-multilingual-MiniLM-L12-v2`. Model ini sangat penting untuk mengurangi halusinasi karena ia memahami konteks finansial baik dalam Bahasa Indonesia maupun Bahasa Inggris.

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

# Inisialisasi model embedding multilingual
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Model Embedding Multilingual siap digunakan.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model Embedding Multilingual siap digunakan.


# Step 10: Membangun Persistent Vector Store 1
Agar database tidak hilang saat Colab timeout, kita menyimpannya secara permanen di Google Drive.
Sistem akan otomatis mengecek: jika database sudah ada di Drive, ia akan langsung memuatnya (Loading).
Jika belum ada, sistem baru akan memulai proses pembuatan (Embedding).

In [15]:
from langchain_community.vectorstores import Chroma
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Tentukan folder penyimpanan di Drive Anda
PERSIST_DIRECTORY = "/content/drive/MyDrive/RAG_IDX_PROJECT/chroma_db_idx"

# 3. Logika Persistent Storage
if os.path.exists(PERSIST_DIRECTORY):
    print("✅ Database ditemukan di Google Drive. Sedang memuat database...")
    vector_db = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=embedding_model
    )
    print("✅ Database berhasil dimuat tanpa proses ulang.")
else:
    print("❌ Database tidak ditemukan. Memulai proses embedding (Mohon tunggu)...")
    texts = [chunk['content'] for chunk in loaded_chunks]
    metadatas = [{'ticker': chunk['ticker'], 'source': chunk['source']} for chunk in loaded_chunks]

    vector_db = Chroma.from_texts(
        texts=texts,
        embedding=embedding_model,
        metadatas=metadatas,
        persist_directory=PERSIST_DIRECTORY
    )
    print(f"✅ Database berhasil dibuat dan disimpan di: {PERSIST_DIRECTORY}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Database ditemukan di Google Drive. Sedang memuat database...
✅ Database berhasil dimuat tanpa proses ulang.


/tmp/ipykernel_1173/1319064912.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(
